In [30]:
import networkExpansionPy.lib as ne
import numpy as np
import pandas as pd
import pickle as pickle
import seaborn as sns
import matplotlib.pyplot as plt
from scipy.stats import spearmanr,pearsonr,mannwhitneyu
from statsmodels.stats.multitest import multipletests
plt.rcParams['font.family'] = 'Arial'

import warnings
from scipy.sparse import (spdiags, SparseEfficiencyWarning, csc_matrix,
    csr_matrix, isspmatrix, dok_matrix, lil_matrix, bsr_matrix)
warnings.simplefilter('ignore',SparseEfficiencyWarning)

def load_data(network_name):
    with open(ne.asset_path  + "../metabolic_networks/" + network_name,"rb") as filehandler:
        metabolism = pickle.load(filehandler)
    return metabolism

# load the metabolism object from the repo

model_id = "metabolism.v8.01May2023.pkl"
metabolism_v8 = pd.read_pickle('../assets/models/' + model_id)
cids = metabolism_v8.coenzymes



In [31]:
def recouple_p_donors(metabolism, reaction_list, old_pair, new_pair, suffix="_gATP"):
    """Replace a phosphate-donor CID pair in metabolism.network.

    Parameters
    ----------
    metabolism : GlobalMetabolicNetwork
    reaction_list : KEGG IDs ('R00200') and/or full network IDs ('R00200_gATP_v1')
    old_pair : (charged, uncharged) CIDs currently in those reactions
               v8 gATP kinases: ('C00013', 'C00009')  # PPi, Pi
    new_pair : (charged, uncharged) CIDs to write
               UTP/UDP: ('C00075', 'C00015')
    suffix : only recouple reaction IDs containing this string; None = all matches
    """
    old_charged, old_uncharged = old_pair
    new_charged, new_uncharged = new_pair

    rn = metabolism.network["rn"].astype(str)
    bases = {r.split("_")[0] for r in reaction_list}
    hit = rn.isin(reaction_list) | rn.str.split("_").str[0].isin(bases)
    if suffix is not None:
        hit = hit & rn.str.contains(suffix, regex=False)

    hit = hit.to_numpy()
    cid = metabolism.network["cid"].to_numpy(copy=True)
    n_ch = int((hit & (cid == old_charged)).sum())
    n_un = int((hit & (cid == old_uncharged)).sum())
    cid[hit & (cid == old_charged)] = new_charged
    cid[hit & (cid == old_uncharged)] = new_uncharged
    metabolism.network = metabolism.network.assign(cid=cid)

    touched = sorted(pd.unique(metabolism.network["rn"].to_numpy()[hit]))
    print(f"recoupled {len(touched)} reactions: {touched}")
    print(f"  {old_charged} -> {new_charged}: {n_ch} rows")
    print(f"  {old_uncharged} -> {new_uncharged}: {n_un} rows")
    return metabolism


kinase_rns = ["R06836", "R00200", "R01523"]

#kinase_rns = ['R00200', 'R01053', 'R01523', 'R04144', 'R04208', 'R04463', 'R04591', 'R06836', 'R06974', 'R06975']
metabolism_v8 = recouple_p_donors(
    metabolism_v8,
    reaction_list=kinase_rns,
    old_pair=("C00013", "C00009"),  # PPi / Pi currently in gATP kinases
    new_pair=("C00075", "C00015"),  # UTP / UDP
    #new_pair=("C00002","C00008")
)

# sanity check: gATP kinases should now carry UTP/UDP, not PPi/Pi
_k = metabolism_v8.network[
    metabolism_v8.network.rn.str.contains("_gATP")
    & metabolism_v8.network.rn.str.split("_").str[0].isin(kinase_rns)
]
print(_k.groupby("rn")["cid"].apply(lambda s: sorted(s.unique())).to_dict())

recoupled 5 reactions: ['R00200_gATP_v1', 'R00200_gATP_v2', 'R01523_gATP_v1', 'R01523_gATP_v2', 'R06836_gATP']
  C00013 -> C00075: 10 rows
  C00009 -> C00015: 10 rows
{'R00200_gATP_v1': ['C00015', 'C00022', 'C00074', 'C00075', 'Z00069'], 'R00200_gATP_v2': ['C00015', 'C00022', 'C00074', 'C00075', 'Z00026', 'Z00029'], 'R01523_gATP_v1': ['C00015', 'C00075', 'C00199', 'C01182', 'Z00030'], 'R01523_gATP_v2': ['C00015', 'C00075', 'C00199', 'C01182', 'Z00029'], 'R06836_gATP': ['C00015', 'C00075', 'C00119', 'C01151']}


In [33]:
seed_set = pd.read_csv('../assets/seed_set.csv')['ID'].tolist() #+ ["C00183"]
#seed_df = pd.read_csv('assets/seed_set.csv')
#seed_set = seed_df[~seed_df.Type.isin(["Organic carbon"])].ID.tolist()
#seed_set = seed_set + ["C00010"]

print('There were {N} compounds in this seed set...'.format(N=len(seed_set)))

ne_cpds,ne_rxns = metabolism_v8.expand(seed_set,algorithm='trace')
print('There were {N} compounds produced in this expansion!'.format(N=len(ne_cpds)))

# parse data into dataframes
rn_df = pd.DataFrame(pd.Series(ne_rxns),columns = ['iteration'])
rn_df.reset_index(inplace=True)
rn_df.columns = ['rn','direction','iteration']
rn_df['rn_kegg']= rn_df['rn'].apply(lambda x: x.split('_')[0])


There were 70 compounds in this seed set...
There were 4315 compounds produced in this expansion!


In [28]:
from itertools import combinations

# reload so we are not stacking on the kinase-only recouple
model_id = "metabolism.v8.01May2023.pkl"
metabolism_v8 = pd.read_pickle("../assets/models/" + model_id)
seed_set = pd.read_csv("../assets/seed_set.csv")["ID"].tolist()

kinase_rns = ["R00200", "R01523", "R06836"]
ligase_rns = ["R01053", "R04144", "R04208", "R04463", "R04591", "R06974", "R06975"]
all_rns = kinase_rns + ligase_rns

OLD_CHARGED, OLD_UNCHARGED = "C00013", "C00009"  # PPi / Pi
NEW_CHARGED, NEW_UNCHARGED = "C00002", "C00008"  # ATP / ADP
LARGE = 1000

net0 = metabolism_v8.network.copy()
rn = net0["rn"].astype(str)
base = rn.str.split("_").str[0]
is_gatp = rn.str.contains("_gATP", regex=False)
cid0 = net0["cid"].to_numpy()
masks = {r: ((base == r) & is_gatp).to_numpy() for r in all_rns}


def run_swap(swapped, stage):
    hit = np.zeros(len(net0), dtype=bool)
    for r in swapped:
        hit |= masks[r]
    cid = cid0.copy()
    cid[hit & (cid == OLD_CHARGED)] = NEW_CHARGED
    cid[hit & (cid == OLD_UNCHARGED)] = NEW_UNCHARGED
    metabolism_v8.network = net0.assign(cid=cid)
    nec, ner = metabolism_v8.expand(seed_set, algorithm="naive")
    row = {
        "stage": stage,
        "n_swapped": len(swapped),
        "n_kinases_to_ATP": sum(r in kinase_rns for r in swapped),
        "n_ligases_to_ATP": sum(r in ligase_rns for r in swapped),
        "swapped": ",".join(swapped),
        "n_compounds": len(nec),
        "n_reactions": len(ner),
        "large": len(nec) >= LARGE,
    }
    for r in all_rns:
        row[r] = int(r in swapped)
    print(f"{stage:12s}  swapped={','.join(swapped) or '()':40s}  n_cpd={len(nec)}")
    return row


rows = []

# --- stage 1: baseline + each reaction alone ---
rows.append(run_swap((), "baseline"))
for r in all_rns:
    rows.append(run_swap((r,), "single"))

singles = pd.DataFrame(rows)
singles["class"] = singles["swapped"].map(
    lambda x: "kinase" if x in kinase_rns else ("ligase" if x in ligase_rns else "none")
)
essential = singles.loc[(singles.stage == "single") & (~singles.large), "swapped"].tolist()
nonessential = [r for r in all_rns if r not in essential]
print("\nessential (collapse when recoupled to ATP):", essential)
print("nonessential:", nonessential)

# --- stage 2: only combos of nonessentials (skip any combo that includes an essential) ---
combo_rows = []
for n in range(2, len(nonessential) + 1):  # n=0,1 already done
    for swapped in combinations(nonessential, n):
        combo_rows.append(run_swap(swapped, "combo"))

results = pd.concat([singles, pd.DataFrame(combo_rows)], ignore_index=True)
results

baseline      swapped=()                                        n_cpd=4315
single        swapped=R00200                                    n_cpd=4315
single        swapped=R01523                                    n_cpd=4315
single        swapped=R06836                                    n_cpd=4315
single        swapped=R01053                                    n_cpd=4315
single        swapped=R04144                                    n_cpd=437
single        swapped=R04208                                    n_cpd=440
single        swapped=R04463                                    n_cpd=439
single        swapped=R04591                                    n_cpd=443
single        swapped=R06974                                    n_cpd=438
single        swapped=R06975                                    n_cpd=446

essential (collapse when recoupled to ATP): ['R04144', 'R04208', 'R04463', 'R04591', 'R06974', 'R06975']
nonessential: ['R00200', 'R01523', 'R06836', 'R01053']
combo         swappe

,stage,n_swapped,n_kinases_to_ATP,n_ligases_to_ATP,swapped,n_compounds,n_reactions,large,R00200,R01523,R06836,R01053,R04144,R04208,R04463,R04591,R06974,R06975,class
0,baseline,0,0,0,,4315,12534,True,0,0,0,0,0,0,0,0,0,0,none
1,single,1,1,0,R00200,4315,12534,True,1,0,0,0,0,0,0,0,0,0,kinase
2,single,1,1,0,R01523,4315,12534,True,0,1,0,0,0,0,0,0,0,0,kinase
3,single,1,1,0,R06836,4315,12534,True,0,0,1,0,0,0,0,0,0,0,kinase
4,single,1,0,1,R01053,4315,12534,True,0,0,0,1,0,0,0,0,0,0,ligase
5,single,1,0,1,R04144,437,1257,False,0,0,0,0,1,0,0,0,0,0,ligase
6,single,1,0,1,R04208,440,1263,False,0,0,0,0,0,1,0,0,0,0,ligase
7,single,1,0,1,R04463,439,1261,False,0,0,0,0,0,0,1,0,0,0,ligase
8,single,1,0,1,R04591,443,1269,False,0,0,0,0,0,0,0,1,0,0,ligase
9,single,1,0,1,R06974,438,1259,False,0,0,0,0,0,0,0,0,1,0,ligase


In [34]:
import pandas as pd
import numpy as np

# --- setup: fresh model, kinase-only UTP/UDP recouple ---
model_id = "metabolism.v8.01May2023.pkl"
metabolism_v8 = pd.read_pickle("../assets/models/" + model_id)
seed_set = pd.read_csv("../assets/seed_set.csv")["ID"].tolist()

kinase_rns = ["R06836", "R00200", "R01523"]
metabolism_v8 = recouple_p_donors(
    metabolism_v8,
    reaction_list=kinase_rns,
    old_pair=("C00013", "C00009"),
    new_pair=("C00075", "C00015"),
)

# --- traced expansion ---
ne_cpds, ne_rxns = metabolism_v8.expand(seed_set, algorithm="trace")
print(f"compounds: {len(ne_cpds)}")

# compound timeline
cpd_iter = pd.Series(ne_cpds, name="iteration").rename_axis("cid").reset_index()
names = pd.read_csv("../assets/cid2name.csv")
cpd_iter = cpd_iter.merge(names, on="cid", how="left")

# reaction timeline
rxn_iter = (
    pd.Series(ne_rxns, name="iteration")
    .rename_axis(["rn", "direction"])
    .reset_index()
)
rxn_iter["rn_kegg"] = rxn_iter["rn"].str.split("_").str[0]

net = metabolism_v8.network

def reaction_stoich(rn, direction):
    d = net[(net.rn == rn) & (net.direction == direction)]
    subs = d.loc[d.s < 0, ["cid", "s"]].groupby("cid")["s"].sum().abs()
    prods = d.loc[d.s > 0, ["cid", "s"]].groupby("cid")["s"].sum()
    sub_str = " + ".join(f"{names.set_index('cid').loc[c,'cid_name']} [{c}]" for c in subs.index)
    prod_str = " + ".join(f"{names.set_index('cid').loc[c,'cid_name']} [{c}]" for c in prods.index)
    return f"{sub_str}  ->  {prod_str}"

# --- when does ATP appear? ---
ATP = "C00002"
atp_iter = ne_cpds[ATP]
print(f"\nATP first appears at iteration {atp_iter}")
print("ATP in seed?", ATP in seed_set)

# nucleotide / phosphate timeline (the story around ATP)
watch = [
    "C00009", "C00013", "C00119", "C03090", "C00147", "C00212",
    "C00020", "C00002", "C00008", "C00075", "C00015",
]
timeline = (
    cpd_iter[cpd_iter.cid.isin(watch)]
    .sort_values("iteration")[["iteration", "cid", "cid_name"]]
)
print("\nNucleotide / phosphate timeline:")
display(timeline)

# compounds entering the same iteration as ATP
print(f"\nCompounds first appearing at iteration {atp_iter}:")
display(
    cpd_iter[cpd_iter.iteration == atp_iter]
    .sort_values("cid_name")[["cid", "cid_name"]]
)

# --- reactions that PRODUCE ATP in the network ---
atp_prod = net[(net.cid == ATP) & (net.s > 0)][["rn", "direction"]].drop_duplicates()
atp_rxns = rxn_iter.merge(atp_prod, on=["rn", "direction"], how="inner").sort_values("iteration")

print(f"\nATP-producing reactions that fired: {len(atp_rxns)}")
print(f"First ATP-producing iteration: {atp_rxns['iteration'].min()}")

# first batch (usually many fire together)
first_atp_iter = atp_rxns["iteration"].min()
first_atp_rxns = atp_rxns[atp_rxns.iteration == first_atp_iter].copy()
first_atp_rxns["stoich"] = first_atp_rxns.apply(
    lambda r: reaction_stoich(r.rn, r.direction), axis=1
)
print(f"\nReactions producing ATP at iteration {first_atp_iter}:")
display(first_atp_rxns[["rn", "rn_kegg", "direction", "iteration", "stoich"]])

# --- reactions firing in the few iterations before ATP ---
window = 3
lo, hi = first_atp_iter - window, first_atp_iter
print(f"\nAll reactions fired in iterations {lo}-{hi}:")
pre_atp_rxns = rxn_iter[(rxn_iter.iteration >= lo) & (rxn_iter.iteration <= hi)].copy()
pre_atp_rxns["stoich"] = pre_atp_rxns.apply(
    lambda r: reaction_stoich(r.rn, r.direction), axis=1
)
display(pre_atp_rxns.sort_values(["iteration", "rn_kegg"])[
    ["iteration", "rn", "rn_kegg", "direction", "stoich"]
])

# --- flag gATP / UTP kinase usage ---
gatp_fired = rxn_iter[rxn_iter.rn.str.contains("_gATP")].sort_values("iteration")
print("\nAll _gATP reactions fired (with iteration):")
display(
    gatp_fired.assign(
        class_=gatp_fired.rn_kegg.map(
            lambda x: "kinase" if x in kinase_rns else "ligase"
        )
    )[["iteration", "rn", "rn_kegg", "direction", "class_"]]
)

# UTP kinase timing specifically
print("\nUTP-coupled kinase _gATP reactions:")
display(gatp_fired[gatp_fired.rn_kegg.isin(kinase_rns)])

recoupled 5 reactions: ['R00200_gATP_v1', 'R00200_gATP_v2', 'R01523_gATP_v1', 'R01523_gATP_v2', 'R06836_gATP']
  C00013 -> C00075: 10 rows
  C00009 -> C00015: 10 rows
compounds: 4315

ATP first appears at iteration 23
ATP in seed? False

Nucleotide / phosphate timeline:


,iteration,cid,cid_name
37,0,C00009,Orthophosphate
70,1,C00013,Diphosphate
232,9,C03090,5-Phosphoribosylamine
320,12,C00119,5-Phospho-alpha-D-ribose 1-diphosphate
453,21,C00147,Adenine
454,21,C00212,Adenosine
458,22,C00020,AMP
465,23,C00002,ATP
470,23,C00008,ADP
1448,40,C00015,UDP



Compounds first appearing at iteration 23:


,cid,cid_name
463,C02353,"2',3'-Cyclic AMP"
464,C22092,5'-O-Phosphonoadenylyl-(3'->5')-adenosine
470,C00008,ADP
467,C00301,ADP-ribose
465,C00002,ATP
469,C00054,"Adenosine 3',5'-bisphosphate"
462,C00857,Deamino-NAD+
461,C03794,"N6-(1,2-Dicarboxyethyl)-AMP"
468,C03431,S-Inosyl-L-homocysteine
466,C03539,S-Ribosyl-L-homocysteine



ATP-producing reactions that fired: 574
First ATP-producing iteration: 23

Reactions producing ATP at iteration 23:


...

,iteration,rn,rn_kegg,direction,stoich
1273,20,R00720_v3,R00720,reverse,Diphosphate [C00013] + IMP [C00130] + Mg [Z000...
1274,20,R00720_v1,R00720,reverse,Diphosphate [C00013] + IMP [C00130] + Ni [Z000...
1276,20,R00720_v4,R00720,reverse,Diphosphate [C00013] + IMP [C00130] + M2 [Z000...
1282,20,R00720_v2,R00720,reverse,Diphosphate [C00013] + IMP [C00130] + Mn [Z000...
1270,20,R00961_v1,R00961,reverse,Orthophosphate [C00009] + IMP [C00130] + Mn [Z...
...,...,...,...,...,...
1356,23,R10836,R10836,forward,Orthophosphate [C00009] + AMP [C00020] -> Ad...
1394,23,R11535_v1,R11535,reverse,"AMP [C00020] + D-Ribose 1,5-bisphosphate [C011..."
1390,23,R12342,R12342,reverse,AMP [C00020] + O-Phospho-L-serine [C01005] ->...
1386,23,R12346_v1,R12346,reverse,AMP [C00020] + Mn [Z00030] -> H2O [C00001] +...



All _gATP reactions fired (with iteration):


,iteration,rn,rn_kegg,direction,class_
487,9,R01053_gATP,R01053,forward,ligase
596,10,R04144_gATP_v1,R04144,forward,ligase
602,10,R01053_gATP,R01053,reverse,ligase
642,11,R06974_gATP,R06974,forward,ligase
650,11,R04144_gATP_v1,R04144,reverse,ligase
837,12,R06974_gATP,R06974,reverse,ligase
917,13,R04463_gATP,R04463,forward,ligase
1058,14,R04463_gATP,R04463,reverse,ligase
1060,14,R04208_gATP,R04208,forward,ligase
1157,15,R04208_gATP,R04208,reverse,ligase



UTP-coupled kinase _gATP reactions:


,rn,direction,iteration,rn_kegg
4983,R01523_gATP_v2,reverse,41,R01523
4989,R00200_gATP_v1,reverse,41,R00200
4993,R01523_gATP_v1,reverse,41,R01523
4998,R06836_gATP,reverse,41,R06836
5036,R00200_gATP_v2,reverse,41,R00200
5148,R01523_gATP_v2,forward,42,R01523
5154,R00200_gATP_v1,forward,42,R00200
5160,R01523_gATP_v1,forward,42,R01523
5212,R00200_gATP_v2,forward,42,R00200
5264,R06836_gATP,forward,42,R06836


In [ ]:
kinase_gATP = [
    "R06836_gATP",
    "R00200_gATP_v1", "R00200_gATP_v2",
    "R01523_gATP_v1", "R01523_gATP_v2",
]
m = pd.read_pickle("../assets/models/metabolism.v8.01May2023.pkl")
m.network = m.network[~m.network.rn.isin(kinase_gATP)]
nec, ner = m.expand(seed_set, algorithm="naive")
print(len(nec), len(ner))

4315 12524


In [38]:
from collections import defaultdict, deque




names = pd.read_csv("../assets/cid2name.csv")
name = dict(zip(names.cid, names.cid_name))
net = metabolism_v8.network
seed = set(seed_set)

# index producing reactions by cid
prod_rows = net[net.s > 0][["rn", "direction", "cid"]].drop_duplicates()
prod_rows = prod_rows.merge(rn_df, on=["rn", "direction"], how="inner")

def stoich(rn, direction):
    g = net[(net.rn == rn) & (net.direction == direction)].groupby("cid")["s"].sum()
    # catalytic: same cid on both sides (net 0 after grouping still possible as +/- rows)
    subs, prods = [], []
    for c, s in g.items():
        label = f"{name.get(c, c)}[{c}]"
        if s < 0:
            subs.append((c, label, ne_cpds.get(c, np.nan)))
        elif s > 0:
            prods.append((c, label))
    return subs, prods


def traceback(start_cid, max_nodes=400):
    """Walk producers of start_cid back to the seed set.

    A node is a compound. Edges are (rn, direction) that first produce it.
    """
    visited = set()
    edges = []  # parent_cid -> child_cid via reaction
    nodes = []

    q = deque([start_cid])
    while q and len(visited) < max_nodes:
        cid = q.popleft()
        if cid in visited:
            continue
        visited.add(cid)
        t = ne_cpds.get(cid, np.nan)
        nodes.append({
            "cid": cid,
            "name": name.get(cid, cid),
            "iteration": t,
            "is_seed": cid in seed,
        })
        if cid in seed or pd.isna(t):
            continue

        # reactions that produce cid in the iteration it first appears
        hits = prod_rows[(prod_rows.cid == cid) & (prod_rows.iteration == t)]
        if hits.empty:
            continue

        for _, r in hits.iterrows():
            subs, prods = stoich(r.rn, r.direction)
            sub_str = " + ".join(f"{lab}(t={it})" for _, lab, it in subs)
            prod_str = " + ".join(lab for _, lab in prods)
            bottleneck = max((it for _, _, it in subs if pd.notna(it)), default=np.nan)
            edges.append({
                "product": cid,
                "product_name": name.get(cid, cid),
                "product_iter": t,
                "rn": r.rn,
                "direction": r.direction,
                "bottleneck_iter": bottleneck,
                "substrates": sub_str,
                "products": prod_str,
            })
            for scid, _, sit in subs:
                if scid not in visited:
                    q.append(scid)

    return pd.DataFrame(nodes), pd.DataFrame(edges)


def print_tree(start_cid, edges, max_depth=12, max_rxns_per_node=3):
    """Pretty-print one producing reaction per compound (latest-substrate / bottleneck)."""
    by_prod = defaultdict(list)
    for _, r in edges.iterrows():
        by_prod[r.product].append(r)

    seen = set()

    def rec(cid, depth):
        if depth > max_depth or cid in seen:
            return
        seen.add(cid)
        t = ne_cpds.get(cid, np.nan)
        pad = "  " * depth
        seed_tag = "  [SEED]" if cid in seed else ""
        print(f"{pad}{name.get(cid, cid)} [{cid}]  t={t}{seed_tag}")
        if cid in seed:
            return
        rxns = by_prod.get(cid, [])
        if not rxns:
            print(f"{pad}  (no producer at first iteration)")
            return
        rxns = sorted(rxns, key=lambda x: (-(x.bottleneck_iter if pd.notna(x.bottleneck_iter) else -1), x.rn))
        for r in rxns[:max_rxns_per_node]:
            print(f"{pad}  via {r.rn} {r.direction}: {r.substrates}  ->  {r.products}")
        # recurse on substrates of the bottleneck reaction only (clearest path)
        r0 = rxns[0]
        subs, _ = stoich(r0.rn, r0.direction)
        # follow latest-appearing substrate first, skip water/H+/metals optionally
        skip = {"C00001", "C00080", "C00009"}  # water, H+, Pi (seed-like background)
        subs = sorted(subs, key=lambda x: -(x[2] if pd.notna(x[2]) else -1))
        for scid, _, _ in subs:
            if scid in skip:
                continue
            rec(scid, depth + 1)

    rec(start_cid, 0)


nodes, edges = traceback("C00020")  # AMP
print(f"{len(nodes)} compounds, {len(edges)} producing reactions in AMP traceback")
display(edges.sort_values(["product_iter", "product"])[
    ["product_iter", "product_name", "rn", "direction", "bottleneck_iter", "substrates", "products"]
])

print("\n==== AMP traceback (bottleneck branch) ====\n")
#print_tree("C00020", edges)

with pd.option_context("display.max_rows", None, "display.max_colwidth", None):
    display(edges.sort_values(["product_iter", "product"]))



47 compounds, 75 producing reactions in AMP traceback


,product_iter,product_name,rn,direction,bottleneck_iter,substrates,products
22,1,Diphosphate,R00004_v1,reverse,0,Orthophosphate[C00009](t=0),H2O[C00001] + Diphosphate[C00013]
23,1,Diphosphate,R00004_v2,reverse,0,Orthophosphate[C00009](t=0),H2O[C00001] + Diphosphate[C00013]
39,1,L-Aspartate,R00490,reverse,0,Ammonia[C00014](t=0) + Fumarate[C00122](t=0),L-Aspartate[C00049]
58,1,Phosphoenolpyruvate,R00345_v1,forward,0,Orthophosphate[C00009](t=0) + Oxaloacetate[C00...,H2O[C00001] + CO2[C00011] + Phosphoenolpyruvat...
59,1,Phosphoenolpyruvate,R00345_v2,forward,0,Orthophosphate[C00009](t=0) + Oxaloacetate[C00...,H2O[C00001] + CO2[C00011] + Phosphoenolpyruvat...
...,...,...,...,...,...,...,...
3,22,AMP,R00183_v1,reverse,21,Orthophosphate[C00009](t=0) + Adenosine[C00212...,H2O[C00001] + AMP[C00020]
4,22,AMP,R00183_v2,reverse,21,Orthophosphate[C00009](t=0) + Adenosine[C00212...,H2O[C00001] + AMP[C00020]
5,22,AMP,R00183_v3,reverse,21,Orthophosphate[C00009](t=0) + Adenosine[C00212...,H2O[C00001] + AMP[C00020]
6,22,AMP,R00183_v4,reverse,21,Orthophosphate[C00009](t=0) + Adenosine[C00212...,H2O[C00001] + AMP[C00020]



==== AMP traceback (bottleneck branch) ====



,product,product_name,product_iter,rn,direction,bottleneck_iter,substrates,products
22,C00013,Diphosphate,1,R00004_v1,reverse,0,Orthophosphate[C00009](t=0),H2O[C00001] + Diphosphate[C00013]
23,C00013,Diphosphate,1,R00004_v2,reverse,0,Orthophosphate[C00009](t=0),H2O[C00001] + Diphosphate[C00013]
39,C00049,L-Aspartate,1,R00490,reverse,0,Ammonia[C00014](t=0) + Fumarate[C00122](t=0),L-Aspartate[C00049]
58,C00074,Phosphoenolpyruvate,1,R00345_v1,forward,0,Orthophosphate[C00009](t=0) + Oxaloacetate[C00036](t=0),H2O[C00001] + CO2[C00011] + Phosphoenolpyruvate[C00074]
59,C00074,Phosphoenolpyruvate,1,R00345_v2,forward,0,Orthophosphate[C00009](t=0) + Oxaloacetate[C00036](t=0),H2O[C00001] + CO2[C00011] + Phosphoenolpyruvate[C00074]
73,C00940,2-Oxoglutaramate,1,R00269,reverse,0,Ammonia[C00014](t=0) + 2-Oxoglutarate[C00026](t=0),H2O[C00001] + 2-Oxoglutaramate[C00940]
52,C19813,threo-3-Hydroxy-D-aspartate,1,R09683_v1,reverse,0,Ammonia[C00014](t=0) + Oxaloacetate[C00036](t=0),threo-3-Hydroxy-D-aspartate[C19813]
53,C19813,threo-3-Hydroxy-D-aspartate,1,R09683_v2,reverse,0,Ammonia[C00014](t=0) + Oxaloacetate[C00036](t=0),threo-3-Hydroxy-D-aspartate[C19813]
54,C19813,threo-3-Hydroxy-D-aspartate,1,R09683_v3,reverse,0,Ammonia[C00014](t=0) + Oxaloacetate[C00036](t=0),threo-3-Hydroxy-D-aspartate[C19813]
55,C19813,threo-3-Hydroxy-D-aspartate,1,R09683_v4,reverse,0,Ammonia[C00014](t=0) + Oxaloacetate[C00036](t=0),threo-3-Hydroxy-D-aspartate[C19813]
